# LSTM-based Exchange Rate Forecasting

This notebook demonstrates how to train an LSTM model using PyTorch to predict exchange rates from an Excel dataset.

## Configuration

Adjust the configuration dictionary below to match your dataset.
- `excel_path`: Path to the Excel file.
- `date_column`: Name of the column containing dates.
- `target_column`: Dependent variable to predict.
- `feature_columns`: Independent variables used as predictors. Include the target if you want to model it with past values.
- `sequence_length`: Number of past time steps used for each training sample.
- `test_size`: Fraction of data reserved for testing.
- `batch_size`, `epochs`, and `learning_rate`: Training hyperparameters.

In [ ]:
import math
from pathlib import Path
from typing import List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, Dataset

CONFIG = {
    'excel_path': Path('data/exchange_rates.xlsx'),  # Update to your file path
    'date_column': 'Date',
    'target_column': 'Close',
    'feature_columns': ['Close', 'Open', 'High', 'Low'],
    'sequence_length': 30,
    'test_size': 0.2,
    'batch_size': 32,
    'epochs': 50,
    'learning_rate': 1e-3,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}
CONFIG

## Load and Inspect Data

In [ ]:
df = pd.read_excel(CONFIG['excel_path'], parse_dates=[CONFIG['date_column']])
df = df.set_index(CONFIG['date_column']).sort_index()
print(f"Data shape: {df.shape}")
df.head()

In [ ]:
# Descriptive statistics
df.describe().T

## Prepare Features and Targets

In [ ]:
features = CONFIG['feature_columns']
target = CONFIG['target_column']

missing_cols = [col for col in features + [target] if col not in df.columns]
if missing_cols:
    raise ValueError(f"Columns not found in dataset: {missing_cols}")

feature_data = df[features].copy()

scaler = StandardScaler()
scaled_features = scaler.fit_transform(feature_data.values)
scaled_df = pd.DataFrame(scaled_features, index=feature_data.index, columns=features)
scaled_df[target] = df[target].values
scaled_df.head()

In [ ]:
def create_sequences(data: pd.DataFrame, feature_cols: List[str], target_col: str, seq_length: int):
    sequences = []
    targets = []
    for i in range(len(data) - seq_length):
        seq = data.iloc[i:i + seq_length][feature_cols].values
        label = data.iloc[i + seq_length][target_col]
        sequences.append(seq)
        targets.append(label)
    return np.array(sequences, dtype=np.float32), np.array(targets, dtype=np.float32)

seq_length = CONFIG['sequence_length']
X, y = create_sequences(scaled_df, features, target, seq_length)
print(f"Sequences shape: {X.shape}")
print(f"Targets shape: {y.shape}")

In [ ]:
split_index = int(len(X) * (1 - CONFIG['test_size']))
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

print(f"Train samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

## Create Dataset and DataLoader

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, sequences: np.ndarray, targets: np.ndarray):
        self.sequences = torch.from_numpy(sequences)
        self.targets = torch.from_numpy(targets)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx]

train_dataset = SequenceDataset(X_train, y_train)
test_dataset = SequenceDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'])

## Define the LSTM Model

In [ ]:
class LSTMRegressor(nn.Module):
    def __init__(self, num_features: int, hidden_size: int = 64, num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=num_features, hidden_size=hidden_size, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, _ = self.lstm(x)
        last_output = output[:, -1, :]
        return self.fc(last_output).squeeze(-1)

model = LSTMRegressor(num_features=len(features)).to(CONFIG['device'])
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])

## Train the Model

In [ ]:
def train(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for sequences, targets in loader:
        sequences = sequences.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model(sequences)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * sequences.size(0)
    return running_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    preds = []
    actuals = []
    with torch.no_grad():
        for sequences, targets in loader:
            sequences = sequences.to(device)
            targets = targets.to(device)
            outputs = model(sequences)
            loss = criterion(outputs, targets)
            running_loss += loss.item() * sequences.size(0)
            preds.extend(outputs.cpu().numpy())
            actuals.extend(targets.cpu().numpy())
    avg_loss = running_loss / len(loader.dataset)
    return avg_loss, np.array(preds), np.array(actuals)

history = {'train_loss': [], 'test_loss': []}
for epoch in range(1, CONFIG['epochs'] + 1):
    train_loss = train(model, train_loader, criterion, optimizer, CONFIG['device'])
    test_loss, _, _ = evaluate(model, test_loader, criterion, CONFIG['device'])
    history['train_loss'].append(train_loss)
    history['test_loss'].append(test_loss)
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d}: Train Loss = {train_loss:.6f}, Test Loss = {test_loss:.6f}")

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['test_loss'], label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training History')
plt.legend()
plt.show()

## Evaluate on Test Set

In [ ]:
test_loss, test_preds, test_actuals = evaluate(model, test_loader, criterion, CONFIG['device'])
rmse = math.sqrt(mean_squared_error(test_actuals, test_preds))
mae = mean_absolute_error(test_actuals, test_preds)
print(f"Test MSE: {test_loss:.6f}")
print(f"Test RMSE: {rmse:.6f}")
print(f"Test MAE: {mae:.6f}")

## Plot Predictions vs Actuals

In [ ]:
test_indices = scaled_df.index[seq_length + split_index:]
plt.figure(figsize=(12, 5))
plt.plot(test_indices, test_actuals, label='Actual')
plt.plot(test_indices, test_preds, label='Predicted')
plt.xlabel('Date')
plt.ylabel(target)
plt.title('Actual vs Predicted Exchange Rates')
plt.legend()
plt.tight_layout()
plt.show()

## Save the Model (Optional)

In [ ]:
model_path = Path('models/lstm_exchange_rate.pth')
model_path.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    'model_state_dict': model.state_dict(),
    'scaler_mean': scaler.mean_,
    'scaler_scale': scaler.scale_,
    'config': CONFIG,
}, model_path)
print(f"Model saved to {model_path.resolve()}")